In [0]:
df_gold_user=spark.readStream.format("delta").table("spotify.silver.dim_user")


In [0]:
df_gold_user.createOrReplaceTempView("dim_user")

In [0]:
%sql
select * from spotify.silver.dim_user

In [0]:
# %sql
# merge into spotify.silver.dim_user_scd1 as s1
# using dim_user s2
# on s1.user_id = s2.user_id
# when matched and s2.updated_at > s1.updated_at then
# update *
# when not matched then
# insert *


In [0]:
def eachBatch(df, batchId):

    df.createOrReplaceTempView("dim_user_g")

    spark.sql("""
        MERGE INTO spotify.silver.dim_user_scd1 s1
        USING dim_user_g s2
        ON s1.user_id = s2.user_id

        WHEN MATCHED
             AND s2.updated_at > s1.updated_at
        THEN UPDATE SET *

        WHEN NOT MATCHED
        THEN INSERT *
    """)

In [0]:
df_gold_user.writeStream.format("delta").foreachBatch(eachBatch).trigger(availableNow=True).option("checkpointLocation","abfss://destination@projectadlanju.dfs.core.windows.net/dim_user/chekpoint_scd1").start()

In [0]:
%sql
select * from spotify.silver.dim_user_scd1

In [0]:
df_stream=spark.readStream.format("delta").table("spotify.silver.fact_stream")

In [0]:
%sql
create table if not exists spotify.silver.fact_stream_scd2 like spotify.silver.fact_stream

In [0]:
%sql
alter table spotify.silver.fact_stream_scd2 add columns(
is_current string)

In [0]:
%sql
select * from spotify.silver.fact_stream

In [0]:
from pyspark.sql.functions import lit

In [0]:
def factBatch(df,batchId):
    df=df.withColumn("is_current",lit("Y"))
    df.createOrReplaceTempView("stream")
    spark.sql("""
              merge into spotify.silver.fact_stream_scd2 s1
              using stream s2
              on s1.stream_id=s2.stream_id
              when matched and s1.is_current='Y'
              then update set
              s1.is_current='N',
              s1.end_date=current_timestamp()
              when not matched 
              then insert *
             
              """)
    spark.sql("""
             MERGE INTO spotify.silver.fact_stream_scd2 s1
USING stream s2
ON s1.stream_id = s2.stream_id
AND s1.is_current='Y'

WHEN NOT MATCHED THEN
INSERT (
    stream_id,
    user_id,
    track_id,
    date_key,
    listen_duration,
    device_type,
    stream_timestamp,
    is_current,
    start_date,
    end_date,
    updated_at,
    _rescued_data
)
VALUES (
    s2.stream_id,
    s2.user_id,
    s2.track_id,
    s2.date_key,
    s2.listen_duration,
    s2.device_type,
    s2.stream_timestamp,
    'Y',
    current_timestamp(),
    NULL,
    s2.updated_at,
    s2._rescued_data
)
              """)


    

In [0]:
df_stream.writeStream.format("delta").foreachBatch(factBatch).trigger(availableNow=True).option("checkpointLocation","abfss://destination@projectadlanju.dfs.core.windows.net/fact_stream/chekpoint_stream").start()